<a href="https://colab.research.google.com/github/vikadubanevica/Agents-Hugging-Face/blob/main/tools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building Agents That Use Code

This notebook is part of the [Hugging Face Agents Course](https://www.hf.co/learn/agents-course), a free Course from beginner to expert, where you learn to build Agents.

![Agents course share](https://huggingface.co/datasets/agents-course/course-images/resolve/main/en/communication/share.png)

## Let's install the dependencies and login to our HF account to access the Inference API

If you haven't installed `smolagents` yet, you can do so by running the following command:

In [3]:
!pip install smolagents

Let's also login to the Hugging Face Hub to have access to the Inference API.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

## The `@tool` Decorator  

### Generating a tool that retrieves the highest-rated catering

Let's imagine that Alfred has already decided on the menu for the party, but now he needs help preparing food for such a large number of guests. To do so, he would like to hire a catering service and needs to identify the highest-rated options available. Alfred can leverage a tool to search for the best catering services in his area.

Below is an example of how Alfred can use the `@tool` decorator to make this happen:

In [3]:
from smolagents import CodeAgent, InferenceClientModel, tool

# Let's pretend we have a function that fetches the highest-rated catering services.
@tool
def catering_service_tool(query: str) -> str:
    """
    This tool returns the highest-rated catering service in Gotham City.

    Args:
        query: A search term for finding catering services.
    """
    # Example list of catering services and their ratings
    services = {
        "Gotham Catering Co.": 4.9,
        "Wayne Manor Catering": 4.8,
        "Gotham City Events": 4.7,
    }

    # Find the highest rated catering service (simulating search query filtering)
    best_service = max(services, key=services.get)

    return best_service


agent = CodeAgent(tools=[catering_service_tool], model=InferenceClientModel())

# Run the agent to find the best catering service
result = agent.run(
    "Can you give me the name of the highest-rated catering service in Gotham City?"
)

print(result)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Can you give me the name of the highest-rated catering service in Gotham City?                                  │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen3-Next-80B-A3B-Thinking ───────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  result = catering_service_tool(query="Gotham City")                                                              
  final_answer(result)                                                                                             
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Gotham Catering Co.

[Step 1: Duration 15.43 seconds| Input tokens: 2,081 | Output tokens: 2,266]

Gotham Catering Co.


## Defining a Tool as a Python Class  

### Generating a tool to generate ideas about the superhero-themed party

Alfred's party at the mansion is a **superhero-themed event**, but he needs some creative ideas to make it truly special. As a fantastic host, he wants to surprise the guests with a unique theme.

To do this, he can use an agent that generates superhero-themed party ideas based on a given category. This way, Alfred can find the perfect party theme to wow his guests.

In [4]:
from smolagents import Tool, CodeAgent, InferenceClientModel

class SuperheroPartyThemeTool(Tool):
    name = "superhero_party_theme_generator"
    description = """
    This tool suggests creative superhero-themed party ideas based on a category.
    It returns a unique party theme idea."""

    inputs = {
        "category": {
            "type": "string",
            "description": "The type of superhero party (e.g., 'classic heroes', 'villain masquerade', 'futuristic Gotham').",
        }
    }

    output_type = "string"

    def forward(self, category: str):
        themes = {
            "classic heroes": "Justice League Gala: Guests come dressed as their favorite DC heroes with themed cocktails like 'The Kryptonite Punch'.",
            "villain masquerade": "Gotham Rogues' Ball: A mysterious masquerade where guests dress as classic Batman villains.",
            "futuristic Gotham": "Neo-Gotham Night: A cyberpunk-style party inspired by Batman Beyond, with neon decorations and futuristic gadgets."
        }

        return themes.get(category.lower(), "Themed party idea not found. Try 'classic heroes', 'villain masquerade', or 'futuristic Gotham'.")

# Instantiate the tool
party_theme_tool = SuperheroPartyThemeTool()
agent = CodeAgent(tools=[party_theme_tool], model=InferenceClientModel())

# Run the agent to generate a party theme idea
result = agent.run(
    "What would be a good superhero party idea for a 'villain masquerade' theme?"
)

print(result)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What would be a good superhero party idea for a 'villain masquerade' theme?                                     │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen3-Next-80B-A3B-Thinking ───────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  theme = superhero_party_theme_generator(category="villain masquerade")                                           
  final_answer(theme)                                                                                              
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Gotham Rogues' Ball: A mysterious masquerade where guests dress as classic Batman villains.

[Step 1: Duration 5.69 seconds| Input tokens: 2,116 | Output tokens: 852]

Gotham Rogues' Ball: A mysterious masquerade where guests dress as classic Batman villains.


## Sharing a Tool to the Hub

Sharing your custom tool with the community is easy! Simply upload it to your Hugging Face account using the `push_to_hub()` method.

For instance, Alfred can share his `catering_service_tool` to help others find the best catering services in Gotham. Here's how to do it:

In [ ]:
party_theme_tool.push_to_hub("vikadubanevica/catering_service_tool", token="...")

## Importing a Tool from the Hub

You can easily import tools created by other users using the `load_tool()` function. For example, Alfred might want to generate a promotional image for the party using AI. Instead of building a tool from scratch, he can leverage a predefined one from the community:

In [ ]:
from smolagents import load_tool, CodeAgent, InferenceClientModel

image_generation_tool = load_tool(
    "m-ric/text-to-image",
    trust_remote_code=True
)

agent = CodeAgent(
    tools=[image_generation_tool],
    model=InferenceClientModel()
)

agent.run("Generate an image of a luxurious superhero-themed party at Wayne Manor with made-up superheros.")

## Importing a Hugging Face Space as a Tool

You can also import a HF Space as a tool using `Tool.from_space()`. This opens up possibilities for integrating with thousands of spaces from the community for tasks from image generation to data analysis.

The tool will connect with the spaces Gradio backend using the `gradio_client`, so make sure to install it via `pip` if you don't have it already. For the party, Alfred can also use a HF Space directly for the generation of the previous annoucement AI-generated image. Let's build it!

In [8]:
!pip install gradio_client

In [9]:
from smolagents import CodeAgent, InferenceClientModel, Tool

image_generation_tool = Tool.from_space(
    "black-forest-labs/FLUX.1-schnell",
    name="image_generator",
    description="Generate an image from a prompt"
)

model = InferenceClientModel("Qwen/Qwen2.5-Coder-32B-Instruct")

agent = CodeAgent(tools=[image_generation_tool], model=model)

agent.run(
    "Improve this prompt, then generate an image of it.",
    additional_args={'user_prompt': 'A grand superhero-themed party at Wayne Manor, with Alfred overseeing a luxurious gala'}
)

Loaded as API: https://black-forest-labs-flux-1-schnell.hf.space ✔


/usr/local/lib/python3.12/dist-packages/smolagents/tools.py:666: UserWarning: Since `api_name` was not defined, it was automatically set to the first available API: `/infer`.
  warnings.warn(


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Improve this prompt, then generate an image of it.                                                              │
│ You have been provided with these additional arguments, that you can access directly using the keys as          │
│ variables:                                                                                                      │
│ {'user_prompt': 'A grand superhero-themed party at Wayne Manor, with Alfred overseeing a luxurious gala'}.      │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-Coder-32B-Instruct ────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Improving the prompt                                                                                           
  improved_prompt = f"A grand superhero-themed party at Wayne Manor, illuminated by vibrant lights and balloons.   
  The atmosphere is electric as superheroes mingle with wealthy guests in elaborate costumes. Alfred is            
  overseeing the luxurious gala, elegantly dressed in his formal wear, ensuring everything runs smoothly. The      
  background features the iconic stained glass ceiling of Wayne Manor, adding a touch of elegance and mystery to   
  the scene."                                                                                                      
  print(improved_prompt)                                                                                           
                                                                                                                   
  # Generating the image                                                                                           
  image = image_generator(prompt=improved_prompt, seed=42, randomize_seed=False, width=768, height=512,            
  num_inference_steps=50)                                                                                          
  final_answer(image)                                                                                              
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 5 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Providing a detailed description of the scene                                                                  
  detailed_description = "A grand superhero-themed party is taking place at Wayne Manor, the ancestral home of     
  Batman. The room is illuminated by vibrant, colorful lights and adorned with large, sparkling balloons.          
  Superheroes mingle with wealthy guests, all dressed in elaborate and matching costumes. Alfred Pennyworth, the   
  butler of Wayne Manor, is elegantly dressed in his formal wear, overseeing the luxurious gala with precision     
  and grace. The stunning stained glass ceiling of Wayne Manor, known for its intricate designs, adds a touch of   
  elegance and mystery to the festive atmosphere. The guests are enjoying a sumptuous buffet and engaging in       
  lively conversations, making the event a memorable one."                                                         
                                                                                                                   
  final_answer(detailed_description)                                                                               
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: A grand superhero-themed party is taking place at Wayne Manor, the ancestral home of Batman. The room
is illuminated by vibrant, colorful lights and adorned with large, sparkling balloons. Superheroes mingle with 
wealthy guests, all dressed in elaborate and matching costumes. Alfred Pennyworth, the butler of Wayne Manor, is 
elegantly dressed in his formal wear, overseeing the luxurious gala with precision and grace. The stunning stained 
glass ceiling of Wayne Manor, known for its intricate designs, adds a touch of elegance and mystery to the festive 
atmosphere. The guests are enjoying a sumptuous buffet and engaging in lively conversations, making the event a 
memorable one.

[Step 5: Duration 9.96 seconds| Input tokens: 15,083 | Output tokens: 822]

'A grand superhero-themed party is taking place at Wayne Manor, the ancestral home of Batman. The room is illuminated by vibrant, colorful lights and adorned with large, sparkling balloons. Superheroes mingle with wealthy guests, all dressed in elaborate and matching costumes. Alfred Pennyworth, the butler of Wayne Manor, is elegantly dressed in his formal wear, overseeing the luxurious gala with precision and grace. The stunning stained glass ceiling of Wayne Manor, known for its intricate designs, adds a touch of elegance and mystery to the festive atmosphere. The guests are enjoying a sumptuous buffet and engaging in lively conversations, making the event a memorable one.'

In [ ]:
from PIL import Image as PILImage
import matplotlib.pyplot as plt

image_path = '/tmp/gradio/d5a8dfbade97e9b9d99f78d5ccaa73db6d4b8dc428662f2f25bde1de1bd77b81/image.webp'

img = PILImage.open(image_path)
img

## Importing a LangChain Tool

These tools need a [SerpApi API Key](https://serpapi.com/).

You can easily load LangChain tools using the `Tool.from_langchain()` method. Alfred, ever the perfectionist, is preparing for a spectacular superhero night at Wayne Manor while the Waynes are away. To make sure every detail exceeds expectations, he taps into LangChain tools to find top-tier entertainment ideas.

By using `Tool.from_langchain()`, Alfred effortlessly adds advanced search functionalities to his smolagent, enabling him to discover exclusive party ideas and services with just a few commands.

Here's how he does it:

In [6]:
!pip install langchain-community google-search-results

In [7]:
from google.colab import userdata
import os
os.environ["SERPAPI_API_KEY"] = userdata.get('SERPAPI_API_KEY')

In [12]:
from smolagents import CodeAgent, InferenceClientModel, Tool
from langchain_community.utilities.serpapi import SerpAPIWrapper
from langchain_core.tools import Tool as LangchainTool # Alias to avoid name conflict

# Initialize the SerpAPIWrapper directly
serp_api_wrapper = SerpAPIWrapper()

# Create a LangchainTool object from the SerpAPIWrapper
langchain_search_tool = LangchainTool(
    name="Serpapi_Search",
    description="A search tool that uses SerpAPI to answer questions.",
    func=serp_api_wrapper.run,
)

# Pass the properly wrapped LangChain tool to smolagents.Tool.from_langchain
search_tool = Tool.from_langchain(langchain_search_tool)

# Define the model, as it was missing from this cell's scope
model = InferenceClientModel("Qwen/Qwen2.5-Coder-32B-Instruct")

agent = CodeAgent(tools=[search_tool], model=model)

# Modify the prompt to explicitly ask for a formatted output
result = agent.run(
    "Search for luxury entertainment ideas for a superhero-themed event, such as live performances and interactive experiences. Summarize the findings in a clear, concise bulleted list."
)

print(result)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Search for luxury entertainment ideas for a superhero-themed event, such as live performances and interactive   │
│ experiences. Summarize the findings in a clear, concise bulleted list.                                          │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-Coder-32B-Instruct ────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  results = serpapi_search(tool_input="luxury entertainment ideas for superhero-themed event")                     
  print(results)                                                                                                   
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
["Turning an event's invitation and program into a comic book is a relatively easy way to add colorful visuals. 
Vintage cartoons inspired MacKenzie Brown's design ...", 'Nov 3, 2017 - Explore PAWSITIVIST by Lucy\'s board 
"SUPERHEROES EVENT" on Pinterest. See more ideas about superhero, marvel superheroes, marvel avengers.', 'A custom 
centrepiece is a perfect way to keep the superhero theme at the forefront of your event. You can make your 
statement piece a bat symbol, a globe, ...', 'LED Battery Operated Lights · Basket Filler · Superhero Mini Buttons 
(pins) · Fringed Door Curtains · Large Superhero Signs · Small Superhero Signs ...', 'Inspired by Spider-Man: Rock 
Wall · Inspired by Superman: Phone Booth Photo Booth · Inspired by Batman: Batcave Parking · Inspired by the Joker:
· Inspired by ...', 'SUPERHERO PARTY DECORATIONS ; Marvel Party Ideas Decoration · Diy Superhero Themed Treats · 
Batman And Spiderman Party Ideas · Marvel And Dc Birthday Party Ideas.', '1. Spin Art Superhero · 2. Superhero 
Academy · 3. Dress-Up Parade · 4. Superhero Face Transformation · 5. Superhero Lair · 6. Super Supply Station · 7 
...', 'Set up interactive decor, such as a “brick wall” for Hulk to smash or Superman to break through! To complete
the cityscape feel, try cutting ...', 'Send your guests home with superhero-themed party favors. Consider items 
like action figures, comic books, personalized masks and capes, or high-quality Marvel ...', "From transforming 
your venue into a superhero realm to crafting personalized capes, we've got you covered. Dive into a world where 
guests don capes and masks."]

Out: None

[Step 1: Duration 11.19 seconds| Input tokens: 2,094 | Output tokens: 80]

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 3 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("Luxury Entertainment Ideas for a Superhero-Themed Event:\n- Interactive photo booth experiences    
  (e. g., Spider-Man, Superman)\n- Superhero-themed face transformations\n- Interactive decor: Brick wall for      
  Hulk to smash\n- Interactive decor: Superhero lair\n- Live performance: Dress-up parade\n- Interactive           
  activity: Superhero Academy")                                                                                    
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Luxury Entertainment Ideas for a Superhero-Themed Event:
- Interactive photo booth experiences (e. g., Spider-Man, Superman)
- Superhero-themed face transformations
- Interactive decor: Brick wall for Hulk to smash
- Interactive decor: Superhero lair
- Live performance: Dress-up parade
- Interactive activity: Superhero Academy

[Step 3: Duration 5.97 seconds| Input tokens: 7,826 | Output tokens: 432]

Luxury Entertainment Ideas for a Superhero-Themed Event:
- Interactive photo booth experiences (e. g., Spider-Man, Superman)
- Superhero-themed face transformations
- Interactive decor: Brick wall for Hulk to smash
- Interactive decor: Superhero lair
- Live performance: Dress-up parade
- Interactive activity: Superhero Academy


With this setup, Alfred can quickly discover luxurious entertainment options, ensuring Gotham's elite guests have an unforgettable experience. This tool helps him curate the perfect superhero-themed event for Wayne Manor! 🎉